# Fine-tune Depth Anything on Mars, or not?

The team plan says: fine-tune Depth Anything V2 on Navcam photos with stereo depth. Notebook 01 showed that **no training**
plus a two-number scaling already gets about 5% error. This notebook runs the fair test:

* **Train** on 37 stereo frames from two Curiosity sites (sol 520 and sol 1400).
* **Test** on 19 frames from a third site it never saw (sol 1100).
* Score both models the same way: scale with 200 stereo pixels per photo, measure error on all the others.

**Free GPU:** in VS Code, *Select Kernel → Colab → New Colab Server → T4 GPU*, then Run All.
The saved outputs below come from a 4 GB laptop GPU; a Colab T4 runs it faster and can take more epochs and frames.

## 0 · Setup and settings

In [ ]:
import sys, re, time, json, urllib.request, concurrent.futures as cf
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt, cv2
IN_COLAB = 'google.colab' in sys.modules
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if dev == 'cuda' else 'none')
BASE = 'https://planetarydata.jpl.nasa.gov/img/data/msl/MSLNAV_1XXX/DATA_V1/'
TRAIN_SITES = [('SOL00520', '0251238'), ('SOL01400', '0552222')]   # (sol folder, site+drive)
TEST_SITES = [('SOL01100', '0492902')]
SIZE = 518            # Depth Anything's native input size
EPOCHS, BATCH, LR = 12, 4, 5e-5
plt.rcParams.update({'figure.dpi': 90})

## 1 · Download the stereo frames
For each site: every left-Navcam photo (`RASLF`) that has a matching 3D file (`XYZLF`). About 15 MB per frame.

In [ ]:
LOCAL = {('SOL01100', '0492902'): Path('../data/raw/sol1100'), }
def frame_list(sol, sd, cache):
    files = sorted(p.name for p in cache.glob(f'NLB_*XYZLF{sd}*.IMG')) if cache.exists() else []
    if not files:
        html = urllib.request.urlopen(BASE + sol + '/', timeout=300).read().decode(errors='ignore')
        files = sorted(set(re.findall(rf'NLB_\d{{9}}XYZLF{sd}NCAM\d{{5}}M1\.IMG', html)))
    return [f.replace('XYZLF', '{}LF').replace('.IMG', '') for f in files]

def fetch(sol, name, folder):
    p = folder / name
    if not p.exists() or p.stat().st_size == 0:
        urllib.request.urlretrieve(BASE + sol + '/' + name, p)
    return p

def get_site(sol, sd):
    folder = LOCAL.get((sol, sd), Path('train_raw'))
    folder.mkdir(exist_ok=True, parents=True)
    stems = frame_list(sol, sd, folder)
    jobs = [(sol, s.format(k) + ext, folder) for s in stems for k in ('RAS', 'XYZ') for ext in ('.LBL', '.IMG')]
    with cf.ThreadPoolExecutor(6) as ex:
        list(ex.map(lambda j: fetch(*j), jobs))
    return [(folder, s) for s in stems]

t0 = time.time()
train_frames = [f for s in TRAIN_SITES for f in get_site(*s)]
test_frames = [f for s in TEST_SITES for f in get_site(*s)]
print(f'{len(train_frames)} training frames, {len(test_frames)} test frames ({time.time() - t0:.0f} s)')

## 2 · Reader, camera model, and turning each frame into (photo, inverse depth, mask)

In [ ]:
def read_label(path):
    text = re.sub(r"/\*.*?\*/", "", Path(path).read_text(errors="ignore"))
    out, stack, key, buf = [], [], None, ""
    for line in text.splitlines():
        if key is not None:
            buf += " " + line.strip()
            if buf.count("(") <= buf.count(")") and buf.count('"') % 2 == 0:
                out.append((tuple(stack), key, buf.strip())); key = None
            continue
        m = re.match(r"\s*([\^A-Z0-9_:]+)\s*=\s*(.*)$", line)
        if not m: continue
        k, v = m.group(1), m.group(2).strip()
        if k in ("GROUP", "OBJECT"): stack.append(v)
        elif k in ("END_GROUP", "END_OBJECT"): stack.pop()
        elif v.count("(") > v.count(")") or v.count('"') % 2: key, buf = k, v
        else: out.append((tuple(stack), k, v))
    return out
get = lambda lbl, key, g=None: next(v for p, k, v in lbl if k == key and (g is None or g in p))
nums = lambda v: tuple(float(x) for x in re.findall(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", v.split("<")[0]))

def read_product(folder, stem):
    lbl = read_label(folder / (stem + ".LBL"))
    rec, start = int(get(lbl, "RECORD_BYTES")), int(nums(get(lbl, "^IMAGE").split(",")[-1])[0])
    L, S, B = (int(get(lbl, k, "IMAGE")) for k in ("LINES", "LINE_SAMPLES", "BANDS"))
    dt = {"MSB_INTEGER": ">i2", "IEEE_REAL": ">f4"}[get(lbl, "SAMPLE_TYPE", "IMAGE")]
    a = np.fromfile(folder / (stem + ".IMG"), dtype=dt, count=L * S * B, offset=(start - 1) * rec)
    return a.astype(np.float32).reshape(B, L, S), lbl

def quat(q):
    s, x, y, z = q
    return np.array([[1-2*(y*y+z*z), 2*(x*y-s*z), 2*(x*z+s*y)], [2*(x*y+s*z), 1-2*(x*x+z*z), 2*(y*z-s*x)],
                     [2*(x*z-s*y), 2*(y*z+s*x), 1-2*(x*x+y*y)]])

MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
def prepare(folder, stem):
    img, _ = read_product(folder, stem.format('RAS')); img = img[0]
    xyz, lbl = read_product(folder, stem.format('XYZ'))
    g = "GEOMETRIC_CAMERA_MODEL_PARMS"
    Cc, Aa = (np.array(nums(get(lbl, f"MODEL_COMPONENT_{i}", g))) for i in (1, 2))
    R = quat(nums(get(lbl, "ORIGIN_ROTATION_QUATERNION", "ROVER_COORD_SYSTEM_PARMS")))
    t = np.array(nums(get(lbl, "ORIGIN_OFFSET_VECTOR", "ROVER_COORD_SYSTEM_PARMS")))
    P = (np.moveaxis(xyz, 0, -1).astype(float) - t) @ R          # rover frame
    valid = (np.abs(xyz).sum(0) > 0) & ~((P[..., 0] > -1.6) & (P[..., 0] < 1.9) & (np.abs(P[..., 1]) < 1.5) & (P[..., 2] < -0.25))
    depth = np.where(valid, (P - Cc) @ Aa, np.nan).astype(np.float32)
    lo, hi = np.percentile(img[img > 0], [0.5, 99.5]) if (img > 0).any() else (0, 1)
    g8 = np.clip((img - lo) / max(hi - lo, 1), 0, 1).astype(np.float32)
    x = cv2.resize(g8, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    x = ((np.stack([x] * 3, 0) - MEAN[:, None, None]) / STD[:, None, None]).astype(np.float32)
    inv = np.where(valid, 1 / np.maximum(depth, 0.1), 0).astype(np.float32)
    w = cv2.resize(valid.astype(np.float32), (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    inv_s = cv2.resize(inv, (SIZE, SIZE), interpolation=cv2.INTER_AREA) / np.maximum(w, 1e-6)
    return dict(x=x, inv=inv_s.astype(np.float32), mask=w > 0.99, depth_full=depth, name=stem.format('RAS'))

t0 = time.time()
train = [d for d in (prepare(*f) for f in train_frames) if d['mask'].mean() > 0.05]
test = [d for d in (prepare(*f) for f in test_frames) if np.isfinite(d['depth_full']).mean() > 0.05]
print(f'usable: {len(train)} train, {len(test)} test frames (frames with under 5% stereo dropped) in {time.time() - t0:.0f} s')
fig, ax = plt.subplots(1, 4, figsize=(14, 3.6))
for i, d in enumerate(train[:2]):
    ax[2 * i].imshow(d['x'][0], cmap='gray'); ax[2 * i].set_title('training photo'); ax[2 * i].axis('off')
    ax[2 * i + 1].imshow(np.where(d['mask'], d['inv'], np.nan), cmap='turbo'); ax[2 * i + 1].set_title('target: stereo 1/depth'); ax[2 * i + 1].axis('off')
plt.show()

## 3 · The model and the loss
* Load Depth Anything V2 Small. **Freeze the encoder** (the part that understands images) and train only the neck + head
  (the part that paints depth), so a small Mars dataset can't wreck what it already knows.
* **Scale-and-shift-invariant loss**: centre and scale both maps the same way, then compare. It cares about the *shape*
  of the depth map, which is exactly what we later fix with the 200-pixel scaling.

In [ ]:
from transformers import AutoModelForDepthEstimation
MODEL = 'depth-anything/Depth-Anything-V2-Small-hf'
def load():
    return AutoModelForDepthEstimation.from_pretrained(MODEL).to(dev)

def ssi_loss(pred, tgt, mask):
    loss = 0
    for p, g, m in zip(pred, tgt, mask):
        p, g = p[m], g[m]
        p = (p - p.median()) / (p - p.median()).abs().mean().clamp_min(1e-6)
        g = (g - g.median()) / (g - g.median()).abs().mean().clamp_min(1e-6)
        loss = loss + (p - g).abs().mean()
    return loss / len(pred)

model = load()
for p_ in model.backbone.parameters(): p_.requires_grad = False
trainable = [p_ for p_ in model.parameters() if p_.requires_grad]
print(f'trainable: {sum(p_.numel() for p_ in trainable) / 1e6:.1f} M of {sum(p_.numel() for p_ in model.parameters()) / 1e6:.1f} M parameters')

## 4 · Train

In [ ]:
X = torch.tensor(np.stack([d['x'] for d in train])); Y = torch.tensor(np.stack([d['inv'] for d in train]))
M = torch.tensor(np.stack([d['mask'] for d in train]))
opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=dev == 'cuda')
history, t0 = [], time.time()
model.train()
for ep in range(EPOCHS):
    perm, tot = torch.randperm(len(X)), 0
    for i in range(0, len(X), BATCH):
        b = perm[i:i + BATCH]
        x, y, m = X[b].to(dev), Y[b].to(dev), M[b].to(dev)
        if torch.rand(1) < 0.5:   # mirror flip: free extra data
            x, y, m = x.flip(-1), y.flip(-1), m.flip(-1)
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
            pred = model(pixel_values=x).predicted_depth
        loss = ssi_loss(pred.float(), y, m)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(b)
    history.append(tot / len(X))
    print(f'epoch {ep + 1:2d}/{EPOCHS}  loss {history[-1]:.4f}  ({time.time() - t0:.0f} s)')
plt.figure(figsize=(5, 3)); plt.plot(range(1, EPOCHS + 1), history, 'o-'); plt.xlabel('epoch'); plt.ylabel('training loss'); plt.title('lower = better fit to training sites'); plt.show()
finetuned = model.eval()

## 5 · The fair test: a site neither model trained on
For every test photo: predict, upsample to 1024 × 1024, fit `s` and `t` on 200 random stereo pixels,
then measure error on all the other stereo pixels. Same procedure for both models.

In [ ]:
zero_shot = load().eval()
BINS = [(0, 4), (4, 8), (8, 12), (12, 20), (20, 60)]

def fit(method, r, z):          # three ways to turn the model's output into metres (see the course, lesson 8)
    if method == 'plain':
        s, b = np.linalg.lstsq(np.stack([r, np.ones_like(r)], 1), 1 / z, rcond=None)[0]
        return lambda R_: 1 / np.maximum(s * R_ + b, 1e-3)
    if method == 'weighted':
        s, b = np.linalg.lstsq(np.stack([r * z, z], 1), np.ones_like(r), rcond=None)[0]
        return lambda R_: 1 / np.maximum(s * R_ + b, 1e-3)
    k, c = np.polyfit(np.log(np.maximum(r, 1e-3)), np.log(z), 1)
    return lambda R_: np.exp(c) * np.maximum(R_, 1e-3) ** k

def evaluate(net, data, seed=0):
    rs, rows = np.random.default_rng(seed), []
    for d in data:
        with torch.no_grad(), torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
            out = net(pixel_values=torch.tensor(d['x'])[None].to(dev)).predicted_depth.float()
        rel = torch.nn.functional.interpolate(out[None], size=(1024, 1024), mode='bicubic', align_corners=False)[0, 0].cpu().numpy()
        gt = np.where(d['depth_full'] < 60, d['depth_full'], np.nan); valid = np.isfinite(gt)
        ys, xs = np.nonzero(valid); pick = rs.choice(len(ys), 200, replace=False)
        r_a, z_a = rel[ys[pick], xs[pick]], gt[ys[pick], xs[pick]]
        chk = np.arange(200) >= 150      # 150 pixels to fit, 50 to pick the best fit
        best = min(('plain', 'weighted', 'power'),
                   key=lambda mth: np.mean(np.abs(fit(mth, r_a[~chk], z_a[~chk])(r_a[chk]) - z_a[chk]) / z_a[chk]))
        m = valid.copy(); m[ys[pick], xs[pick]] = False
        row = {'frame': d['name'][4:13], 'fit': best}
        for tag, mth in [('plain', 'plain'), ('chosen', best)]:
            pred = fit(mth, r_a, z_a)(rel)
            err = np.abs(pred[m] - gt[m]) / gt[m]
            row[f'{tag}:all'], row[f'{tag}:within25'] = err.mean(), np.mean(err < 0.25)
            for lo, hi in BINS:
                k = (gt[m] >= lo) & (gt[m] < hi)
                row[f'{tag}:{lo}-{hi} m'] = err[k].mean() if k.sum() > 500 else np.nan
        rows.append(row)
    return rows

r0, r1 = evaluate(zero_shot, test), evaluate(finetuned, test)
cols = ['all', 'within25'] + [f'{lo}-{hi} m' for lo, hi in BINS]
print('Average error on the TEST site (sol 1100). "within25" = share of pixels within 25% of stereo.\n')
print(f"{'':32s}" + ''.join(f'{c:>10s}' for c in cols))
for name, rows in [('no training', r0), ('fine-tuned', r1)]:
    for tag in ['plain', 'chosen']:
        label = f'{name}, {"one plain fit" if tag == "plain" else "fit chosen per photo"}'
        print(f'{label:32s}' + ''.join(f'{np.nanmean([r[f"{tag}:{c}"] for r in rows]):>10.1%}' for c in cols))
for tag in ['plain', 'chosen']:
    better = sum(a[f'{tag}:all'] > b[f'{tag}:all'] for a, b in zip(r0, r1))
    print(f'\nwith {"one plain fit" if tag == "plain" else "the fit chosen per photo"}: fine-tuned was better on {better} of {len(test)} test photos')

## 6 · Picture the result

In [ ]:
labels = [f'{lo}-{hi} m' for lo, hi in BINS]
x = np.arange(len(labels))
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for j, (name, rows, tag) in enumerate([('no training, plain fit', r0, 'plain'), ('fine-tuned, plain fit', r1, 'plain'),
                                       ('no training, chosen fit', r0, 'chosen'), ('fine-tuned, chosen fit', r1, 'chosen')]):
    ax[0].bar(x - 0.3 + 0.2 * j, [np.nanmean([r[f'{tag}:{l}'] for r in rows]) * 100 for l in labels], 0.2, label=name)
ax[0].set_xticks(x, labels); ax[0].set_ylabel('average error (% of true depth)'); ax[0].set_title('Test site (sol 1100), by distance'); ax[0].legend(fontsize=8)
for tag, mk in [('plain', 'o'), ('chosen', 's')]:
    ax[1].scatter([r[f'{tag}:all'] * 100 for r in r0], [r[f'{tag}:all'] * 100 for r in r1], marker=mk,
                  label='one plain fit' if tag == 'plain' else 'fit chosen per photo')
lim = max(max(r['plain:all'] for r in r0 + r1) * 100, 1) * 1.1
ax[1].plot([0, lim], [0, lim], 'k--'); ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].set_xlabel('no training: error per photo (%)'); ax[1].set_ylabel('fine-tuned (%)')
ax[1].set_title('below the line = fine-tuning helped'); ax[1].legend(); plt.show()

## 7 · What to take from this
* Read the table above: the decision is whatever the **test-site** numbers say, not the training loss.
* Fine-tuning needs: photos + stereo depth (NASA has thousands), a frozen encoder, a shape-only loss, and a split by site.
* 37 frames and a dozen epochs is a tiny run. A Colab T4 can take a few hundred frames from more sites; rerun this
  notebook with more `TRAIN_SITES` to see whether the gap grows.
* Whatever wins, the pipeline is the same: predict, scale with stereo pixels, fill holes, build the mesh.